[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/forest_basics.ipynb)

# Random forests, in pictures

Run each cell with **Shift and Enter**. Five tasks, each one number to change,
and the answer written underneath.

A forest is many decision trees, each grown on its own draw of the rows, all
voting on the answer.

## 1. One tree, and how much it moves

Four hundred points in two arcs that run into each other. Sixty of every
hundred are used for growing and forty are held back.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def show(model, X, y, title):
    """The two measurements, the classes, and the boxes the model cuts."""
    step = 0.02
    gx, gy = np.meshgrid(np.arange(X[:, 0].min() - 0.4, X[:, 0].max() + 0.4, step),
                         np.arange(X[:, 1].min() - 0.4, X[:, 1].max() + 0.4, step))
    zone = model.predict(np.c_[gx.ravel(), gy.ravel()]).reshape(gx.shape)

    fig, ax = plt.subplots(figsize=(5.6, 3.6))
    ax.contourf(gx, gy, zone, levels=np.arange(-0.5, len(names) + 0.5, 1.0),
                colors=["#dfe4ec", "#fbeddc", "#e9e9f4"][:len(names)])
    ax.contour(gx, gy, zone, levels=np.arange(0.5, len(names) - 0.5, 1.0),
               colors="#b45309", linewidths=1.0)
    for k in range(len(names)):
        ax.scatter(X[y == k, 0], X[y == k, 1], marker="os^"[k], s=16,
                   color=["#1e3a5f", "#b45309", "#64748b"][k],
                   label=names[k], zorder=3)
    ax.set_xlabel(labels[0])
    ax.set_ylabel(labels[1])
    ax.set_title(title)
    ax.legend(frameon=False, fontsize=8, loc="upper right")
    plt.show()

from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier

X, y = make_moons(n_samples=400, noise=0.28, random_state=0)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.4, random_state=0)
labels = ["first measurement", "second measurement"]
names = ["group 1", "group 2"]

seed = 0
part = np.random.default_rng(seed).choice(len(Xtr), len(Xtr))
one = DecisionTreeClassifier(random_state=0).fit(Xtr[part], ytr[part])

print("this tree gets right:", round(one.score(Xte, yte), 3))
show(one, Xtr, ytr, "one tree, grown on one draw of the rows")

The rows were drawn with replacement, so this tree saw its own version of the
table. The boundary is a staircase of narrow boxes, and several of them exist
to catch a single point.

**Task 1.** Change `seed = 0` to `seed = 3` and run the cell again.

*Answer.* The staircase is drawn somewhere else and the score goes from 0.869
to 0.900. Try `seed = 5` as well, which gives 0.838. Across the first eight
draws this same code scores between 0.838 and 0.900: one tree depends on which
rows it happened to get, and that is what a forest is built to fix.

## 2. Many trees, one vote

A random forest grows many such trees, each on its own draw, and lets them
vote.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=10, random_state=0).fit(Xtr, ytr)

print("ten trees get right:", round(forest.score(Xte, yte), 3))
show(forest, Xtr, ytr, "ten trees, voting")

The narrow boxes are gone. Where one tree drew a hard edge around a single
point, ten trees disagree and the vote lands somewhere smoother.

**Task 2.** Change `n_estimators=10` to `n_estimators=200` and run it again.

*Answer.* The boundary gets smoother still and the score goes from 0.906 to
0.919. Past a few dozen trees the picture keeps tidying up and the answer stops
moving.

## 3. What the trees say about one point

A vote is worth looking at one row at a time.

In [ ]:
row = 0
point = Xte[row:row + 1]

votes = [int(t.predict(point)[0]) for t in forest.estimators_]
counted = {names[k]: votes.count(k) for k in (0, 1)}

print("the point:", point[0].round(2), "  truly:", names[yte[row]])
print("the trees say:", counted)
print("the forest says:", names[forest.predict(point)[0]])

fig, ax = plt.subplots(figsize=(4.4, 2.4))
ax.bar(list(counted), list(counted.values()), color="#1e3a5f")
ax.set_ylabel("trees voting")
plt.show()

Every tree gets one vote and the forest answers with the winner.

**Task 3.** Change `row = 0` to `row = 7` and run it again.

*Answer.* The ten trees split five against five, the forest answers group 1,
and that point truly belongs to group 2. A split vote is the forest saying the
point sits where the two groups overlap, and those are the points it gets
wrong.

## 4. More trees, steadier answer

Growing a forest twice gives two different forests, because the draws are
random. Grow five of each size and look at the worst and the best.

In [ ]:
sizes = [1, 2, 5, 10, 25, 50, 100]
runs = [[RandomForestClassifier(n_estimators=b, random_state=k)
         .fit(Xtr, ytr).score(Xte, yte) for k in range(5)] for b in sizes]

fig, ax = plt.subplots(figsize=(5.4, 3.0))
ax.plot(sizes, [np.mean(r) for r in runs], marker="o", color="#1e3a5f")
ax.fill_between(sizes, [min(r) for r in runs], [max(r) for r in runs],
                color="#dfe4ec")
ax.set_xlabel("trees in the forest")
ax.set_ylabel("share right")
plt.show()

for b, r in zip(sizes, runs):
    print("%3d trees   average %.3f   worst %.3f   best %.3f"
          % (b, np.mean(r), min(r), max(r)))

The line is the average of five forests and the band is how far the worst and
the best of them sat apart.

**Task 4.** Read the three columns and say what more trees bought.

*Answer.* The average climbs from 0.879 at one tree to 0.921 at ten, and then
moves by less than half a point. The five forests of a single tree landed
between 0.850 and 0.919; the five of a hundred landed between 0.919 and 0.925.
More trees buy a better answer first, and a less accidental one after that.

## 5. Which measurement the forest leans on

A forest can say how much each column did for it: how much impurity the splits
on that column removed, added up over every tree. The two curved groups have
columns with no names, so this last picture uses the flowers from the tree
notebook.

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
full = RandomForestClassifier(n_estimators=200, random_state=0)
full.fit(iris.data[:, :], iris.target)

order = np.argsort(full.feature_importances_)
fig, ax = plt.subplots(figsize=(5.4, 2.6))
ax.barh([iris.feature_names[i] for i in order],
        full.feature_importances_[order], color="#1e3a5f")
ax.set_xlabel("share of the impurity removed")
plt.show()

for i in order[::-1]:
    print("%-20s %.3f" % (iris.feature_names[i], full.feature_importances_[i]))

The two petal columns carry almost everything, which is why the tree notebook
drew its pictures with them.

**Task 5.** Change `full.fit(iris.data[:, :], iris.target)` to
`full.fit(iris.data[:, :2], iris.target)` and run it again: that keeps the two
sepal columns and drops the petals.

*Answer.* The bars now show the sepal columns sharing the work, because they
are all the forest has. Importance is a statement about the columns a forest
was given, and says nothing about the ones it never saw.

## What you can say now

A forest grows many trees, each on its own draw of the rows, and answers with
their vote. One tree moves when the rows move; the vote of many moves much
less. The first dozen trees do most of the work, the rest steady the answer,
and the forest can tell you which measurements its trees kept cutting on.